# Image Processing in RAG: Failure → Description Remedy
### Why raster images inside PDFs are invisible to text-only pipelines, and how to fix it

In [1]:
!pip install langchain langchain-community langchain-openai langchain-pinecone pinecone langchain-text-splitters pypdf pymupdf pillow python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, time, base64, pathlib
from dotenv import load_dotenv
load_dotenv()

import fitz  # PyMuPDF
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

PDF_PATH = "Steps_of_UAN_Activation_09Jan2018.pdf"
INDEX_NAME = "mmrag-openai"
NAMESPACE = "uan"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

C:\Users\shiva\AppData\Local\Temp\ipykernel_7236\2577011608.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Naive text-only baseline
`PyPDFLoader` reads only the text layer of the PDF. Every step in this guide is followed by a screenshot of the actual EPFO portal — those screenshots are pure raster images pasted into the source document, so a text loader walks straight past them without ever knowing they exist.

In [3]:
pc = Pinecone()
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # text-embedding-3-small output size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

text_docs = PyPDFLoader(PDF_PATH).load()
text_chunks = splitter.split_documents(text_docs)
for d in text_chunks:
    d.metadata["chunk_type"] = "text"

# Deterministic IDs make this upsert idempotent: re-running this notebook overwrites
# the same vectors in place instead of accumulating duplicate copies in the shared namespace.
text_ids = [f"uan_text_{i}" for i in range(len(text_chunks))]

vs = PineconeVectorStore.from_documents(text_chunks, embedding=embeddings, index_name=INDEX_NAME, namespace=NAMESPACE, ids=text_ids)
print(f"Baseline: {len(text_chunks)} text-only chunks upserted into '{INDEX_NAME}/{NAMESPACE}'")

Baseline: 6 text-only chunks upserted into 'mmrag-openai/uan'


## Step 2: Ask a question the text can't answer
Page 3's "Member Profile" screenshot shows a fully filled-in sample form — a UAN number, a name, a date of birth, and a **Bank Account No.** value. None of that appears anywhere in the surrounding paragraph text; it only exists inside the pasted screenshot.

In [4]:
RAG_PROMPT = """Answer the question using only the following context. If the context does not contain the answer, say so plainly instead of guessing.

Context:
{context}

Question: {question}
Answer:"""

def answer_from_index(question, k=4):
    hits = vs.similarity_search(question, k=k, namespace=NAMESPACE)
    context = "\n\n".join(f"[{h.metadata.get('chunk_type')}] {h.page_content}" for h in hits)
    response = llm.invoke(RAG_PROMPT.format(context=context, question=question)).content.strip()
    return hits, response

question = "In the sample 'Member Profile' screenshot, what Bank Account No. and UAN number are shown?"
hits, response = answer_from_index(question)
print("Retrieved chunk_types:", [h.metadata.get("chunk_type") for h in hits])
print("\nAnswer:\n", response)

Retrieved chunk_types: ['text', 'text', 'text', 'text']

Answer:
 The context does not contain the answer to the question about the Bank Account No. and UAN number shown in the sample 'Member Profile' screenshot.


## Failure: the image was invisible to the pipeline
This isn't a retrieval-ranking problem — no chunk in the index contains this answer, at any rank. The screenshot was never read, so the model either declines to answer or invents a plausible-looking number. Fixing this means reaching into the PDF for the image itself, not tuning the retriever.

## Step 3: Extract the screenshot with PyMuPDF
`fitz` walks each page's embedded image objects directly and writes them back out as standalone files — the exact bytes the PDF stores, no re-rendering.

In [5]:
IMG_DIR = pathlib.Path("extracted_images")
IMG_DIR.mkdir(exist_ok=True)

def extract_images(pdf_path, out_dir, prefix):
    doc = fitz.open(pdf_path)
    paths = []
    for page_index in range(len(doc)):  # 0-indexed internally
        page = doc[page_index]
        for img_i, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base = doc.extract_image(xref)
            ext = base["ext"]
            out_path = out_dir / f"{prefix}_p{page_index + 1}_img{img_i}.{ext}"  # human page number
            out_path.write_bytes(base["image"])
            paths.append((page_index + 1, out_path))
    doc.close()
    return paths

image_paths = extract_images(PDF_PATH, IMG_DIR, prefix="uan")
print(f"Extracted {len(image_paths)} images -> {IMG_DIR}/")

Extracted 8 images -> extracted_images/


## Step 4: Caption each image with a vision-capable LLM
Each image is base64-encoded and sent to `gpt-4o-mini` as an `image_url` content block alongside a text instruction — the same chat model already used for generation, just given eyes for this one call.

In [6]:
def encode_image(path):
    return base64.b64encode(path.read_bytes()).decode("utf-8")

CAPTION_PROMPT = (
    "Describe exactly what this screenshot shows, including any labeled fields and their "
    "visible sample values. Be literal and specific — this description will be the only "
    "record of the image's content."
)

def caption_image(path):
    ext = path.suffix.lstrip(".")
    b64 = encode_image(path)
    message = HumanMessage(content=[
        {"type": "text", "text": CAPTION_PROMPT},
        {"type": "image_url", "image_url": {"url": f"data:image/{ext};base64,{b64}"}},
    ])
    return llm.invoke([message]).content.strip()

image_docs = []
for page_num, path in image_paths:
    caption = caption_image(path)
    image_docs.append(Document(
        page_content=caption,
        metadata={"chunk_type": "image_desc", "page": page_num, "image_path": str(path)},
    ))
    print(f"Page {page_num} ({path.name}): {caption[:120]}...")

Page 1 (uan_p1_img0.jpeg): The screenshot displays a webpage from the Employees' Provident Fund Organisation (EPFO) of India. The top section featu...


Page 1 (uan_p1_img1.jpeg): The screenshot displays a webpage titled "Activate Your UAN" from the Employees' Provident Organisation, India. The top ...


Page 2 (uan_p2_img0.jpeg): The screenshot displays a webpage from the Employees' Provident Fund Organisation of India. At the top, there is a logo ...


Page 2 (uan_p2_img1.jpeg): The screenshot displays a webpage titled "Activate Your UAN" from the Employees' Provident Fund Organisation, India. The...


Page 3 (uan_p3_img0.jpeg): The screenshot displays a webpage from the Employees' Provident Fund Organisation (EPFO) of India. The top section inclu...


Page 3 (uan_p3_img1.jpeg): The screenshot displays a webpage from the Employees' Provident Fund Organization of India. The main section is titled "...


Page 4 (uan_p4_img0.jpeg): The screenshot displays a webpage from the Employees' Provident Fund Organisation (EPFO) of India. 

### Header:
- The t...


Page 4 (uan_p4_img1.jpeg): The screenshot displays a webpage from the Employees' Provident Fund Organisation of India. The top section includes the...


## Step 5: Embed the descriptions into the SAME index
The captions are just text now — they go through the identical embedding model and land in the identical `uan` namespace as the baseline chunks. Nothing downstream needs to know they originated from an image.

In [7]:
# Deterministic IDs (derived from the extracted image filename) keep this upsert
# idempotent across re-runs, same rationale as the text chunk IDs above.
image_ids = [pathlib.Path(d.metadata["image_path"]).stem for d in image_docs]

vs.add_documents(image_docs, namespace=NAMESPACE, ids=image_ids)
print(f"Added {len(image_docs)} image-description chunks into '{INDEX_NAME}/{NAMESPACE}'")

Added 8 image-description chunks into 'mmrag-openai/uan'


## Step 6: Re-run the same question

In [8]:
# This is a shared, persistent namespace, so several near-duplicate "Activate Your UAN"
# screenshots (pages 1-2) and their embeddings outrank the page-3 "Member Profile" caption
# on raw vocabulary overlap with the question. A larger k pulls it into context anyway.
hits, response = answer_from_index(question, k=15)
print("Retrieved chunk_types:", [h.metadata.get("chunk_type") for h in hits])
print("\nAnswer:\n", response)

Retrieved chunk_types: ['text', 'image_desc', 'text', 'image_desc', 'image_desc', 'text', 'image_desc', 'image_desc', 'text', 'text', 'image_desc', 'text', 'image_desc', 'image_desc']

Answer:
 The sample 'Member Profile' screenshot shows the Bank Account No. as "Not Available" and the UAN number as "100099430466."


## Wrap-up
Captioning and re-embedding turned an invisible image into an ordinary text chunk — the fix is cheap and it generalizes to any image-bearing PDF. But notice the ceiling: the answer is now only as good as the caption. A vague or lossy caption loses exactly the detail a sharper method would keep. Notebook 2 tackles that ceiling directly, by embedding the image itself instead of a text description of it.

## Try it yourself
1. Ask a second question whose answer lives in a different screenshot (e.g. the KYC screen on page 4) and confirm the same failure → remedy pattern holds.
2. Reattach the actual retrieved image (via its `image_path` metadata) to the final generation call instead of relying on the caption alone, and compare the answer's precision.
3. Deliberately write a vague caption prompt (e.g. just "describe this image") and see how much retrieval quality degrades.

**Cleanup:** this notebook writes into the shared `mmrag-openai` index under the `uan` namespace only (Notebook 3 uses the same index under `factsheet`). Delete just this namespace when done — deleting the whole index would remove Notebook 3's data too:
```python
pc.Index(INDEX_NAME).delete(delete_all=True, namespace=NAMESPACE)
```

1. Pdf loader
2. Img, Text, Table - Extract
3. Text - Chunk -> Index
4. Img - Store it in a Storage -> Gather Description (LLM) -> Index description with Storage FilePath
5. Table - "HTML/XML File data" -> Gather Description (LLM) -> Index Description with Direct HTML Data
6. Query - Send the query to the index, retrieve text, img_desc, table desc
7. Retrieve Img & Table from the storage path -> Feed it into Final LLM Context